<div dir="rtl" align="right">

# NB22 — انتشار نسخه و بازتولیدپذیری

این نوت‌بوک داده، کد، مدل، تنظیمات، گزارش‌ها و شواهد موجود فاز دوم
را به یک نسخهٔ قابل ردیابی و قابل بازسازی متصل می‌کند.

ورودی اصلی، بستهٔ نهایی NB21 و Release/Handoffهای موجود مؤلفه‌هاست.
خروجی شامل Manifest، Cardها، محدودیت‌ها، Evidence index،
آزمون Restore/Validate-only و نتیجهٔ واقعی Final Release Gate است.

در این مرحله آزمایش‌ها و داوری‌های معتبر قبلی تکرار نمی‌شوند.
تولید دوبارهٔ Golden، Embedding، پاسخ و داوری جمعیت، کار پیش‌فرض NB22 نیست.

موفقیت نهایی به کنترل نسخه‌ها و هش‌ها، بازخوانی مستقل،
بازتولیدپذیری و احراز شرایط پذیرش وابسته است.
سلامت بسته‌بندی، جای PASS کیفیت محصول را نمی‌گیرد.

## نقطهٔ شروع

ابتدا وجود نسخهٔ محلی NB22/NB23 و پنج فایل نهایی C082/C085
بررسی می‌شود. این بررسی فقط‌خواندنی است و هیچ API، اجرای مدل،
تغییر چک‌لیست یا نوشتن Artifact ندارد.

</div>

In [1]:
# NB21-C001 — Centralized standard-library imports for NB21.
# Imports only; no processing, file writes, repository changes, or API calls.

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib import metadata
import unicodedata
import ast
import csv
import hashlib
import itertools
import json
import math
import os
import platform
import random
import re
import statistics
import subprocess
import sys
import tempfile
import time
from urllib.parse import urlsplit, urlunsplit
from dotenv import dotenv_values
import tiktoken
import jsonschema
import requests

<div dir="rtl" align="right">

### NB22-C02 — بررسی ورودی نهایی و نسخه‌های محلی

- ریشه و وضعیت Git محلی ثبت می‌شود.
- وجود NB22/NB23 و پنج فایل مشخص‌شده در تحویل‌نامه بررسی می‌شود.
- هش بایت فایل‌ها با هش گزارش‌شده مقایسه می‌شود.
- ساختار اولیهٔ JSON برای طراحی Reader متناسب با دادهٔ واقعی نمایش داده می‌شود.
- فایل مفقود یا هش متفاوت، خودکار به معنی انجام‌نشدن کار قبلی نیست.
- این سل دریافت کامل بسته یا پذیرش Release را اعلام نمی‌کند.

API تازه، اجرای مدل و نوشتن فایل: صفر.

</div>

In [2]:
# NB22-C02 — Read-only final-handoff preflight
# Scope: local Git reads and five explicit files; no network or writes.

def nb22_c02_run():
    def git(cwd, *args):
        result = subprocess.run(
            ["git", "-C", str(cwd), *args],
            capture_output=True,
            text=True,
            check=False,
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"LOCAL_GIT_READ_FAILED: {' '.join(args)}"
            )
        return result.stdout.strip()

    root = Path(git(Path.cwd(), "rev-parse", "--show-toplevel"))
    roadmap = (
        root / "data/Road_Maps/"
        "RAG_finance_phase2_modular_notebooks_execution_plan.md"
    )
    if not roadmap.is_file():
        raise RuntimeError("EXPECTED_PROJECT_ROADMAP_NOT_FOUND")

    branch = git(root, "branch", "--show-current")
    if branch != "rag-phase2":
        raise RuntimeError(
            f"UNEXPECTED_BRANCH: {branch}; no branch change performed"
        )

    final_dir = (
        "reports/phase2/evaluation/"
        "NB21C.C085.action49_final_summary/"
    )
    handoff_dir = (
        "reports/phase2/evaluation/"
        "NB21C.C082.action46_handoff/"
        "9186bb5a30456486b29d521db1660b944"
        "c72bf8c4fcb9d816cf578b1937ea1d9/"
    )

    expected = [
        (
            "C085_report",
            final_dir,
            "C085.report",
            "a2e7f80edb4a8057a4f184bcc8b319bc"
            "aaac432261c0ecb8ae2cdd2817427dde",
            ".json",
        ),
        (
            "C085_actions",
            final_dir,
            "C085.actions01_49_status",
            "234e1e8326bbaf5e8a06082e47b97ef6"
            "0f60b99b7d355284ca594d9b7dce5a88",
            ".json",
        ),
        (
            "C085_checklist_proposal",
            final_dir,
            "C085.checklist_status_proposal_fa",
            "39d39dd58694ce9187d78cf4de8bc601"
            "825b39742f2c8aebdf2ff4009303c069",
            ".md",
        ),
        (
            "C082_manifest",
            handoff_dir,
            "C082.manifest",
            "212f8f411a5c39b693e108c5ba3100a2"
            "91e40355625e60ab14bae343af88e174",
            ".json",
        ),
        (
            "C082_handoff",
            handoff_dir,
            "C082.handoff_NB22",
            "aeffc73207cc137a492496eb3083d290"
            "88afeab4447bc749faafc6aea397bdec",
            ".json",
        ),
    ]

    def shape(value, depth=0):
        """Show structure without printing report text or private values."""
        if isinstance(value, dict):
            if depth >= 2:
                return {"type": "object", "keys": list(value)}
            return {
                key: shape(item, depth + 1)
                for key, item in value.items()
            }
        if isinstance(value, list):
            return {
                "type": "array",
                "count": len(value),
                "first_item_shape": (
                    shape(value[0], depth + 1) if value else None
                ),
            }
        return {"type": type(value).__name__}

    rows = []
    for role, directory, stem, expected_sha, suffix in expected:
        relative = directory + stem + "." + expected_sha + suffix
        path = root / relative
        row = {
            "role": role,
            "path": relative,
            "expected_sha256_source": "USER_SUPPLIED_HANDOFF",
            "expected_file_bytes_sha256": expected_sha,
        }

        if not path.is_file():
            row["status"] = "NOT_AVAILABLE_AT_EXPLICIT_LOCAL_PATH"
            rows.append(row)
            continue

        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)

        actual_sha = digest.hexdigest()
        row.update({
            "bytes": path.stat().st_size,
            "actual_file_bytes_sha256": actual_sha,
            "hash_matches": actual_sha == expected_sha,
        })

        if actual_sha != expected_sha:
            row["status"] = "HASH_MISMATCH_REQUIRES_RECONCILIATION"
        elif suffix == ".json":
            try:
                with path.open("r", encoding="utf-8") as stream:
                    value = json.load(stream)
                row["status"] = "HASH_MATCH_JSON_READABLE"
                row["json_structure"] = shape(value)
            except (ValueError, UnicodeError) as exc:
                row["status"] = "HASH_MATCH_JSON_READ_FAILED"
                row["error_type"] = type(exc).__name__
        else:
            row["status"] = "HASH_MATCH_CONTENT_NOT_YET_REVIEWED"

        rows.append(row)

    notebook_candidates = sorted({
        path.relative_to(root).as_posix()
        for pattern in ("22*.ipynb", "23*.ipynb")
        for path in (root / "notebooks/phase2").glob(pattern)
        if path.is_file()
    })

    summary = {
        "status": "READONLY_PREFLIGHT_COMPLETED_NOT_ACCEPTANCE",
        "branch": branch,
        "local_head": git(root, "rev-parse", "HEAD"),
        "local_changes": git(
            root, "status", "--porcelain=v1", "--untracked-files=normal"
        ).splitlines(),
        "unmerged_paths": git(
            root, "diff", "--name-only", "--diff-filter=U"
        ).splitlines(),
        "local_nb22_nb23_candidates": notebook_candidates,
        "explicit_files": rows,
        "full_manifest_reference_validation_performed": False,
        "release_acceptance_assessed": False,
        "files_written": 0,
        "model_api_calls": 0,
        "network_calls": 0,
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary


nb22_c02_result = nb22_c02_run()

{
  "status": "READONLY_PREFLIGHT_COMPLETED_NOT_ACCEPTANCE",
  "branch": "rag-phase2",
  "local_head": "271c5a1b6115021dcc538a3a11197a3e255d383e",
  "local_changes": [
    "?? notebooks/phase2/22_release_and_reproducibility.ipynb",
    "?? notebooks/phase2/checklist/NB22_remaining_work_checklist_fa.md"
  ],
  "unmerged_paths": [],
  "local_nb22_nb23_candidates": [
    "notebooks/phase2/22_release_and_reproducibility.ipynb"
  ],
  "explicit_files": [
    {
      "role": "C085_report",
      "path": "reports/phase2/evaluation/NB21C.C085.action49_final_summary/C085.report.a2e7f80edb4a8057a4f184bcc8b319bcaaac432261c0ecb8ae2cdd2817427dde.json",
      "expected_sha256_source": "USER_SUPPLIED_HANDOFF",
      "expected_file_bytes_sha256": "a2e7f80edb4a8057a4f184bcc8b319bcaaac432261c0ecb8ae2cdd2817427dde",
      "bytes": 5047,
      "actual_file_bytes_sha256": "a2e7f80edb4a8057a4f184bcc8b319bcaaac432261c0ecb8ae2cdd2817427dde",
      "hash_matches": true,
      "status": "HASH_MATCH_JSON_READABL